In [ ]:
!pip install "numpy<2.0.0"

In [1]:
!pip uninstall -y cupy cupy-cuda11x cupy-cuda12x

Found existing installation: cupy-cuda12x 14.0.1
Uninstalling cupy-cuda12x-14.0.1:
  Successfully uninstalled cupy-cuda12x-14.0.1


### Batch Processing of Yoga Dataset
This script will iterate through the `dataset` folder, maintaining the category structure for the generated 3D models.

In [1]:
import torch
print(f"Is CUDA available?: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Using CPU. The process will be VERY slow.")

Is CUDA available?: True
GPU detected: NVIDIA GeForce RTX 4060 Laptop GPU


In [4]:
!pip install mediapipe opencv-python-headless pandas plotly scipy -q

In [7]:
!pip uninstall mediapipe -y
!pip install mediapipe==0.10.14

     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ---------------------------------------- 61.0/61.0 kB 3.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
   ---------------------------------------- 0.5/50.8 MB 10.2 MB/s eta 0:00:05
    --------------------------------------- 1.0/50.8 MB 13.3 MB/s eta 0:00:04
    --------------------------------------- 1.0/50.8 MB 13.3 MB/s eta 0:00:04
   - -------------------------------------- 2.0/50.8 MB 10.6 MB/s eta 0:00:05
   - -------------------------------------- 2.2/50.8 MB 9.5 MB/s eta 0:00:06
   - -------------------------------------- 2.5/50.8 MB 7.2 MB/s eta 0:00:07
   -- ------------------------------------- 3.6/50.8 MB 9.2 MB/s eta 0:00:06
   --- ------------------------------------ 4.1/50.8 MB 8.5 MB/s eta 0:00:06
   ---- ----------------------------------- 5.8/50.8 MB 11.2 MB/s eta 0:00:05
   ----- ---------------------------------- 6.9/50.8 MB 12.0 MB/s eta 0:00:04
  

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
captum 0.8.0 requires numpy<2.0, but you have numpy 2.4.4 which is incompatible.
fairlearn 0.13.0 requires scipy<1.16.0,>=1.9.3, but you have scipy 1.17.1 which is incompatible.
music21 9.7.0 requires numpy<2.0.0, but you have numpy 2.4.4 which is incompatible.
numba 0.63.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.4 which is incompatible.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
ultralytics 8.4.18 requires ultralytics-thop>=2.0.18, but you have ultralytics-thop 2.0.9 which is incompatible.


In [8]:
!pip uninstall mediapipe -y

Found existing installation: mediapipe 0.10.14
Uninstalling mediapipe-0.10.14:
  Successfully uninstalled mediapipe-0.10.14


In [5]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial.distance import cosine
from collections import defaultdict

BASE_DIR     = os.getcwd()
DATASET_ROOT = r"C:\Users\User\Desktop\3D vision project\dataset"
POSE_OUTPUT  = os.path.join(BASE_DIR, "pose_results")
os.makedirs(POSE_OUTPUT, exist_ok=True)

# ── Download model if it doesn't exist ────────────────────────────────────────────
import urllib.request
MODEL_PATH = os.path.join(BASE_DIR, "pose_landmarker_heavy.task")
if not os.path.exists(MODEL_PATH):
    print(" Downloading model (~30MB)...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
        "pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task",
        MODEL_PATH
    )
    print(" Model downloaded")
else:
    print(" Model already exists")

# ── Initialize detector ──────────────────────────────────────────────────────
BaseOptions           = mp.tasks.BaseOptions
PoseLandmarker        = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode     = mp.tasks.vision.RunningMode

POSE_OPTIONS = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
POSE_MODEL = PoseLandmarker.create_from_options(POSE_OPTIONS)
print(f" MediaPipe {mp.__version__} — detector ready")

# ── Indices of landmarks (API Tasks uses numeric indices) ────────────────────
LM = {
    "nose":            0,
    "l_shoulder":     11, "r_shoulder":     12,
    "l_elbow":        13, "r_elbow":        14,
    "l_wrist":        15, "r_wrist":        16,
    "l_hip":          23, "r_hip":          24,
    "l_knee":         25, "r_knee":         26,
    "l_ankle":        27, "r_ankle":        28,
    "l_foot":         31, "r_foot":         32,
}

JOINT_PAIRS = [
    ("l_shoulder","l_elbow"), ("l_elbow","l_wrist"),
    ("r_shoulder","r_elbow"), ("r_elbow","r_wrist"),
    ("l_hip","l_knee"),       ("l_knee","l_ankle"),
    ("r_hip","r_knee"),       ("r_knee","r_ankle"),
    ("l_shoulder","l_hip"),   ("r_shoulder","r_hip"),
    ("l_hip","r_hip"),        ("l_shoulder","r_shoulder"),
]

ANGLE_TRIPLETS = [
    # (name, A, vertex, B)
    ("codo_izq",    "l_shoulder", "l_elbow",    "l_wrist"),
    ("codo_der",    "r_shoulder", "r_elbow",    "r_wrist"),
    ("hombro_izq",  "l_elbow",    "l_shoulder", "l_hip"),
    ("hombro_der",  "r_elbow",    "r_shoulder", "r_hip"),
    ("cadera_izq",  "l_shoulder", "l_hip",      "l_knee"),
    ("cadera_der",  "r_shoulder", "r_hip",      "r_knee"),
    ("rodilla_izq", "l_hip",      "l_knee",     "l_ankle"),
    ("rodilla_der", "r_hip",      "r_knee",     "r_ankle"),
    ("tobillo_izq", "l_knee",     "l_ankle",    "l_foot"),
    ("tobillo_der", "r_knee",     "r_ankle",    "r_foot"),
    ("tronco",      "l_shoulder", "l_hip",      "r_hip"),
]

print(" Model configuration complete")

 Model already exists
 MediaPipe 0.10.35 — detector ready
 Model configuration complete


In [6]:
def compute_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cos_ang = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos_ang, -1.0, 1.0)))


def extract_pose(imagen_path):
    img = cv2.imread(imagen_path)
    if img is None:
        return {"ok": False}

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result   = POSE_MODEL.detect(mp_image)

    if not result.pose_landmarks or len(result.pose_landmarks) == 0:
        return {"ok": False}

    landmarks = result.pose_landmarks[0]  # primera persona
    coords = np.array([[l.x, l.y, l.z] for l in landmarks])      # (33, 3)
    vis    = np.array([l.visibility for l in landmarks])          # (33,)

    # Normalise
    center      = coords.mean(axis=0)
    coords_norm = coords - center
    escale      = np.linalg.norm(coords_norm, axis=1).max() + 1e-8
    coords_norm /= escale

    # Articular angles
    angles = {}
    for nombre, a_key, b_key, c_key in ANGLE_TRIPLETS:
        ai, bi, ci = LM[a_key], LM[b_key], LM[c_key]
        if vis[ai] > 0.3 and vis[bi] > 0.3 and vis[ci] > 0.3:
            angles[nombre] = compute_angle(
                coords[ai], coords[bi], coords[ci]
            )
        else:
            angles[nombre] = np.nan

    return {
        "ok":          True,
        "landmarks":   coords_norm,
        "angles":     angles,
        "vector":      coords_norm.flatten(),
        "visibility": vis,
        "path":        imagen_path,
    }


# Metrics
def cosine_similarity(v1, v2):
    return 1 - cosine(v1, v2)

def rmse_angles(ang1, ang2):
    keys = [k for k in ang1 if not np.isnan(ang1[k]) and not np.isnan(ang2.get(k, np.nan))]
    if not keys:
        return np.nan
    return np.sqrt(np.mean([(ang1[k] - ang2[k])**2 for k in keys]))

def mae_angles(ang1, ang2):
    keys = [k for k in ang1 if not np.isnan(ang1[k]) and not np.isnan(ang2.get(k, np.nan))]
    if not keys:
        return np.nan
    return np.mean([abs(ang1[k] - ang2[k]) for k in keys])

def pct_angles_right(ang1, ang2, umbral=15):
    keys = [k for k in ang1 if not np.isnan(ang1[k]) and not np.isnan(ang2.get(k, np.nan))]
    if not keys:
        return np.nan
    return round(100 * sum(abs(ang1[k] - ang2[k]) <= umbral for k in keys) / len(keys), 1)

def error_per_articulation(ang1, ang2):
    return {
        k: round(abs(ang1[k] - ang2[k]), 1)
        for k in ang1
        if not np.isnan(ang1.get(k, np.nan)) and not np.isnan(ang2.get(k, np.nan))
    }

print(" Model functions ready")

 Model functions ready


In [7]:
EXTENSIONS = ('.png', '.jpg', '.jpeg', '.webp')
poses_db   = {}   # {path: pose_dict}
fails     = []

print("── Escaning dataset ──")
image_files = []
for root, dirs, files in os.walk(DATASET_ROOT):
    for f in files:
        if f.lower().endswith(EXTENSIONS):
            image_files.append(os.path.join(root, f))

print(f"Total images found: {len(image_files)}")

for i, img_path in enumerate(image_files):
    # Guardar resultado en .npz para no reprocesar
    cache_name = img_path.replace(os.sep, "_").replace(":", "") + ".npz"
    cache_path = os.path.join(POSE_OUTPUT, cache_name)

    if os.path.exists(cache_path):
        data = np.load(cache_path, allow_pickle=True)
        poses_db[img_path] = {
            "ok":          bool(data["ok"]),
            "landmarks":   data["landmarks"] if bool(data["ok"]) else None,
            "angles":     data["angles"].item() if bool(data["ok"]) else {},
            "vector":      data["vector"] if bool(data["ok"]) else None,
            "path":        img_path,
        }
        continue

    pose = extract_pose(img_path)
    poses_db[img_path] = pose

    if pose["ok"]:
        np.savez(cache_path,
            ok        = np.array(True),
            landmarks = pose["landmarks"],
            angles   = np.array(pose["angles"]),
            vector    = pose["vector"]
        )
        if (i + 1) % 20 == 0:
            print(f"  [{i+1}/{len(image_files)}] processed...")
    else:
        np.savez(cache_path, ok=np.array(False), landmarks=[], angles=np.array({}), vector=[])
        fails.append(img_path)

ok_count = sum(1 for p in poses_db.values() if p["ok"])
print(f"\n Detected poses: {ok_count}/{len(image_files)}")
print(f" NO detection:    {len(fails)}")
if fails:
    print(" Fails:", [os.path.basename(f) for f in fails[:5]], "...")

── Escaning dataset ──
Total images found: 5991
  [20/5991] processed...
  [40/5991] processed...
  [60/5991] processed...
  [100/5991] processed...
  [120/5991] processed...
  [140/5991] processed...
  [160/5991] processed...
  [180/5991] processed...
  [200/5991] processed...
  [220/5991] processed...
  [240/5991] processed...
  [260/5991] processed...
  [280/5991] processed...
  [300/5991] processed...
  [320/5991] processed...
  [340/5991] processed...
  [360/5991] processed...
  [380/5991] processed...
  [400/5991] processed...
  [440/5991] processed...
  [460/5991] processed...
  [480/5991] processed...
  [500/5991] processed...
  [520/5991] processed...
  [580/5991] processed...
  [600/5991] processed...
  [620/5991] processed...
  [640/5991] processed...
  [680/5991] processed...
  [700/5991] processed...
  [740/5991] processed...
  [760/5991] processed...
  [780/5991] processed...
  [800/5991] processed...
  [820/5991] processed...
  [840/5991] processed...
  [860/5991] proces

In [8]:
per_category = defaultdict(list)

for img_path, pose in poses_db.items():
    if not pose["ok"]:
        continue
    category = os.path.basename(os.path.dirname(img_path))
    per_category[category].append(img_path)

for cat in per_category:
    per_category[cat].sort()

# 3 first categories
categories = sorted(per_category.keys())[:3]
print(f"Categories selected: {categories}\n")

data = {}
for cat in categories:
    paths      = per_category[cat]
    instructor = paths[0]
    students   = paths[1:]
    data[cat] = {
        "instructor":      instructor,
        "instructor_pose": poses_db[instructor],
        "students":         students,
        "students_poses":   [poses_db[p] for p in students],
    }
    print(f"[{cat}]")
    print(f"  instructor : {os.path.basename(instructor)}")
    print(f"  students    : {len(students)} imágenes\n")

Categories selected: ['adho mukha svanasana', 'adho mukha vriksasana', 'agnistambhasana']

[adho mukha svanasana]
  instructor : 1. 1.png
  students    : 67 imágenes

[adho mukha vriksasana]
  instructor : 0-0.png
  students    : 53 imágenes

[agnistambhasana]
  instructor : 1-0.png
  students    : 31 imágenes



In [9]:
results = {}

for cat in categories:
    inst_pose = data[cat]["instructor_pose"]
    results[cat] = []

    for student_path, student_pose in zip(data[cat]["students"], data[cat]["students_poses"]):
        name = os.path.basename(student_path)

        cos_sim = cosine_similarity(inst_pose["vector"], student_pose["vector"])
        rmse    = rmse_angles(inst_pose["angles"], student_pose["angles"])
        mae     = mae_angles(inst_pose["angles"], student_pose["angles"])
        pct     = pct_angles_right(inst_pose["angles"], student_pose["angles"], umbral=15)
        errors = error_per_articulation(inst_pose["angles"], student_pose["angles"])

        results[cat].append({
            "student":            name,
            "Similitud coseno ": round(cos_sim, 4),
            "RMSE angles ":    round(rmse, 2) if not np.isnan(rmse) else np.nan,
            "MAE angles ":     round(mae, 2)  if not np.isnan(mae)  else np.nan,
            "% correct (±15°) ": pct,
            "errores_detalle":   errors,
        })
        print(f"[{cat}] {name[:50]:50s}  cos={cos_sim:.4f}  MAE={mae:.1f}°  %ok={pct}%")

print("\nMetrics computed")

[adho mukha svanasana] 1. 5-benefits-of-downward-facing-dog-pose.png       cos=0.9514  MAE=8.6°  %ok=66.7%
[adho mukha svanasana] 10. screen-shot-2017-09-15-at-17.00.06-1024x585.pn  cos=-0.3270  MAE=10.4°  %ok=75.0%
[adho mukha svanasana] 11. yoga_anatomy_using_muscle_awareness_to_lower_y  cos=-0.3034  MAE=7.2°  %ok=100.0%
[adho mukha svanasana] 12. 66a62b8c606fd88e0401b5af0a7cbca7.png            cos=-0.3544  MAE=16.0°  %ok=0.0%
[adho mukha svanasana] 12. downward-facing-dog-pose.png                    cos=-0.2927  MAE=14.3°  %ok=100.0%
[adho mukha svanasana] 13. downward-facing-dog-800x490.png                 cos=0.9504  MAE=7.4°  %ok=83.3%
[adho mukha svanasana] 14. downward-dog-pose-as-done-by-a-dog.png          cos=0.9482  MAE=6.6°  %ok=100.0%
[adho mukha svanasana] 14. untitled_design_33.png                          cos=-0.2788  MAE=9.2°  %ok=100.0%
[adho mukha svanasana] 16. one-leg-downward-facing-dog-yoga-pose.png       cos=-0.3781  MAE=11.1°  %ok=75.0%
[adho mukha svanasana] 1

In [10]:
for cat in categories:
    print(f"\n{'='*70}")
    print(f"  {cat.upper()}  (instructor: {os.path.basename(data[cat]['instructor'])})")
    print(f"{'='*70}")

    df = pd.DataFrame([
        {k: v for k, v in r.items() if k != "errores_detalle"}
        for r in results[cat]
    ]).set_index("student")

    display(df.style
        .background_gradient(subset=["RMSE angles ", "MAE angles "], cmap="RdYlGn_r")
        .background_gradient(subset=["Similitud coseno ", "% correct (±15°) "], cmap="RdYlGn")
        .format({
            "Similitud coseno ":    "{:.4f}",
            "RMSE angles ":        "{:.1f}°",
            "MAE angles ":         "{:.1f}°",
            "% correct (±15°) ":  "{:.1f}%",
        })
        .highlight_min(subset=["MAE angles "], color="#c6efce")
        .highlight_max(subset=["MAE angles "], color="#ffc7ce")
    )


  ADHO MUKHA SVANASANA  (instructor: 1. 1.png)


,Similitud coseno,RMSE angles,MAE angles,% correct (±15°)
student,,,,
1. 5-benefits-of-downward-facing-dog-pose.png,0.9514,12.4°,8.6°,66.7%
10. screen-shot-2017-09-15-at-17.00.06-1024x585.png,-0.3270,12.0°,10.4°,75.0%
11. yoga_anatomy_using_muscle_awareness_to_lower_your_heels_in_downward_facing_dog_pose.png,-0.3034,7.2°,7.2°,100.0%
12. 66a62b8c606fd88e0401b5af0a7cbca7.png,-0.3544,16.0°,16.0°,0.0%
12. downward-facing-dog-pose.png,-0.2927,14.3°,14.3°,100.0%
13. downward-facing-dog-800x490.png,0.9504,9.6°,7.4°,83.3%
14. downward-dog-pose-as-done-by-a-dog.png,0.9482,6.8°,6.6°,100.0%
14. untitled_design_33.png,-0.2788,9.2°,9.2°,100.0%
16. one-leg-downward-facing-dog-yoga-pose.png,-0.3781,11.5°,11.1°,75.0%



  ADHO MUKHA VRIKSASANA  (instructor: 0-0.png)


,Similitud coseno,RMSE angles,MAE angles,% correct (±15°)
student,,,,
1-0.png,-0.1340,89.1°,88.4°,0.0%
17-0.png,0.3556,28.6°,16.5°,66.7%
19-0.png,0.7823,38.5°,27.0°,50.0%
20-0.png,0.5853,42.1°,33.4°,33.3%
25-0.png,-0.1907,26.6°,26.6°,0.0%
26-0.png,0.3749,52.4°,52.4°,0.0%
29-0.png,0.9602,1.7°,1.5°,100.0%
31-0.png,0.6792,30.2°,27.1°,33.3%
32-0.png,-0.1814,47.9°,40.0°,25.0%



  AGNISTAMBHASANA  (instructor: 1-0.png)


,Similitud coseno,RMSE angles,MAE angles,% correct (±15°)
student,,,,
10-0.png,0.4905,38.9°,30.0°,36.4%
11-0.png,0.4498,27.1°,24.5°,33.3%
12-0.png,0.5008,34.8°,21.8°,63.6%
13-1.png,0.7120,24.4°,19.1°,57.1%
14-0.png,0.5604,27.2°,21.4°,33.3%
15-0.png,0.6883,25.7°,20.7°,54.5%
2-0.png,0.5791,36.5°,29.4°,42.9%
20-0.png,0.4445,40.1°,26.4°,55.6%
21-0.png,0.4050,24.9°,20.1°,54.5%


In [11]:
for cat in categories:
    inst_name = os.path.basename(data[cat]["instructor"])
    filas     = results[cat]

    # Construir matriz: filas=alumnos, columnas=articulaciones
    joints = list(filas[0]["errores_detalle"].keys())
    matrix = []
    names  = []
    for r in filas:
        row = [r["errores_detalle"].get(j, np.nan) for j in joints]
        matrix.append(row)
        names.append(r["student"][:40])

    matrix = np.array(matrix, dtype=float)

    fig = go.Figure(data=go.Heatmap(
        z          = matrix,
        x          = joints,
        y          = names,
        colorscale = "RdYlGn_r",
        zmin=0, zmax=45,
        colorbar   = dict(title="Error (°)"),
        hovertemplate = "Student: %{y}<br>Joint: %{x}<br>Error: %{z:.1f}°<extra></extra>"
    ))
    fig.update_layout(
        title   = f"Articular error per student — {cat}<br><sup>instructor: {inst_name}</sup>",
        xaxis_title = "Articulation",
        yaxis_title = "Student",
        height  = max(400, len(names) * 18 + 150),
        margin  = dict(l=250, r=40, t=80, b=80)
    )
    fig.show()

In [13]:
def score(r, max_mae=90, max_rmse=90):
    """Score 0-100: combina MAE, coseno y % correcto."""
    mae_score = max(0, 100 - (r["MAE ángulos ↓"] / max_mae) * 100)
    cos_score = max(0, r["Similitud coseno ↑"] * 100)
    pct_score = r["% correcto (±15°) ↑"] or 0
    return round((mae_score * 0.4 + cos_score * 0.3 + pct_score * 0.3), 1)

for cat in categories:
    print(f"\n🏆 RANKING — {cat}")
    print(f"   instructor: {os.path.basename(data[cat]['instructor'])}\n")

    ranking = sorted(
        results[cat],
        key=lambda r: score(r),
        reverse=True
    )
    for i, r in enumerate(ranking[:10], 1):
        bar   = "█" * int(score(r) / 5)
        print(f"  #{i:2d}  {r['student'][:45]:45s}  Score={score(r):5.1f}  {bar}")


🏆 RANKING — adho mukha svanasana
   instructor: 1. 1.png



KeyError: 'MAE ángulos ↓'

In [ ]:
def draw_skeleton(pose, title, color_points="#1f77b4"):
    lm     = pose["landmarks"]
    traces = []

    for a_key, b_key in JOINT_PAIRS:
        ai, bi = LM[a_key], LM[b_key]
        a, b   = lm[ai], lm[bi]
        traces.append(go.Scatter(
            x=[a[0], b[0]], y=[-a[1], -b[1]],
            mode="lines", line=dict(color="#aaa", width=2),
            showlegend=False, hoverinfo="skip"
        ))

    traces.append(go.Scatter(
        x=lm[:, 0], y=-lm[:, 1],
        mode="markers",
        marker=dict(size=8, color=color_points),
        name=title,
        hovertemplate="%{text}<extra></extra>",
        text=list(LM.keys()) + [""] * (33 - len(LM)),
    ))
    return traces

for cat in categories:
    mejor = sorted(results[cat], key=lambda r: r["MAE angles"])[0]
    mejor_path = next(p for p in data[cat]["students"] if os.path.basename(p) == mejor["student"])
    mejor_pose = poses_db[mejor_path]
    inst_pose  = data[cat]["instructor_pose"]

    fig = make_subplots(rows=1, cols=2,
        subplot_titles=[
            f"Instructor<br>{os.path.basename(data[cat]['instructor'])[:40]}",
            f"Best student (MAE={mejor['MAE angles']:.1f}°)<br>{mejor['student'][:40]}"
        ])

    for t in draw_skeleton(inst_pose,  "Instructor",     "#3266ad"):
        fig.add_trace(t, row=1, col=1)
    for t in draw_skeleton(mejor_pose, "Best student",   "#D85A30"):
        fig.add_trace(t, row=1, col=2)

    fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False)
    fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, scaleanchor="x", scaleratio=1)
    fig.update_layout(
        title=f"Esqueleto 2D — {cat}",
        height=500,
        showlegend=False
    )
    fig.show()

KeyError: 'MAE angles'

- - -

## AUTOENCODERS

- - -

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

#residual block with LayerNorm instead of BatchNorm1d to avoid collapsing batch statistics at inference
class ResBlock(nn.Module):
    def __init__(self, dim):
        super(ResBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),      #LayerNorm is instance-level, safe at batch_size=1 during inference
            nn.GELU(),              #GELU avoids dead-relu saturation in narrow bottlenecks
            nn.Linear(dim, dim),
            nn.LayerNorm(dim)
        )
        self.act = nn.GELU()

    def forward(self, x):
        #additive skip preserves gradient flow and prevents variance extinction
        return self.act(x + self.block(x))


class RobustPoseAE(nn.Module):
    def __init__(self, input_dim=99, latent_dim=64):
        #latent_dim raised to 64: gives enough capacity to represent extreme-pose peripheral geometry
        super(RobustPoseAE, self).__init__()

        #encoder: project -> normalize -> compress with residual stabilization
        self.enc_proj = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.GELU()
        )
        self.enc_res = ResBlock(128)                      #residual block at 128-dim for stable mid-encoding
        self.enc_bottleneck = nn.Linear(128, latent_dim)  #final compression without normalization to keep scale info

        #decoder: expand -> residual refinement -> output (no activation on final layer)
        self.dec_proj = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.LayerNorm(128),
            nn.GELU()
        )
        self.dec_res = ResBlock(128)                      #symmetric residual block on decode side
        self.dec_out = nn.Linear(128, input_dim)          #linear output preserves the full real-valued coordinate range

        #skip connection path: bypasses bottleneck to preserve absolute scale signal directly
        #this is the most important structural fix for limb collapse
        self.skip_proj = nn.Linear(input_dim, input_dim)

    def encode(self, x):
        h = self.enc_proj(x)
        h = self.enc_res(h)
        return self.enc_bottleneck(h)

    def decode(self, z):
        h = self.dec_proj(z)
        h = self.dec_res(h)
        return self.dec_out(h)

    def forward(self, x):
        z = self.encode(x)
        recon = self.decode(z)
        #skip adds the input-projected residual: prevents the AE from predicting a mean pose
        skip = self.skip_proj(x)
        return recon + 0.15 * skip     #0.15 weight keeps skip as a correction hint, not a passthrough

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

#------------------------------------------------------------
#MEDIAPIPE 33-JOINT INDEX ANATOMY MAP
#peripheral limb joints (hands, feet, ankles, wrists) are the
#ones that collapse first under standard MSE — assign them 3x weight
#central torso joints (hips, shoulders, nose) stay at 1x weight
#------------------------------------------------------------

#indices for peripheral extremity joints in the 33-joint schema
PERIPHERAL_JOINT_INDICES = [
    15, 16,         #wrists
    17, 18, 19, 20, #pinky/index knuckles
    21, 22,         #thumbs
    27, 28,         #ankles
    29, 30,         #heels
    31, 32          #foot index
]

#all remaining joints treated as central/torso weight group
CENTRAL_JOINT_INDICES = [i for i in range(33) if i not in PERIPHERAL_JOINT_INDICES]

def build_joint_weight_vector(peripheral_weight=3.0, device="cpu"):
    #expand joint-level weights to cover all 3 coordinates (x, y, z) per joint
    weights = torch.ones(99, dtype=torch.float32)
    for ji in PERIPHERAL_JOINT_INDICES:
        base = ji * 3
        weights[base]     = peripheral_weight   #x
        weights[base + 1] = peripheral_weight   #y
        weights[base + 2] = peripheral_weight   #z
    return weights.to(device)


def weighted_mse(output, target, weights):
    #element-wise squared error scaled by the per-coordinate weight vector
    sq_err = (output - target) ** 2
    return (sq_err * weights).mean()


def per_joint_variance_loss(output, target):
    #reshape to (batch, 33, 3) to compute variance per joint independently
    #this prevents coarse global std from being satisfied by central joints alone
    out_joints = output.view(-1, 33, 3)
    tgt_joints = target.view(-1, 33, 3)
    #variance per joint across the batch dimension: shape (33, 3)
    out_var = out_joints.var(dim=0)
    tgt_var = tgt_joints.var(dim=0)
    #penalize any joint whose output variance is far below the target variance
    #clamp prevents penalizing overconfident joints that are already well-reconstructed
    return torch.clamp(tgt_var - out_var, min=0).mean()


def bone_length_consistency_loss(output, target, joint_pairs):
    #optional structural loss: penalizes segment length distortion between adjacent joints
    #joint_pairs is a list of (idx_a, idx_b) tuples for connected bones
    out_joints = output.view(-1, 33, 3)
    tgt_joints = target.view(-1, 33, 3)
    total = 0.0
    for a, b in joint_pairs:
        out_len = torch.norm(out_joints[:, a] - out_joints[:, b], dim=1)
        tgt_len = torch.norm(tgt_joints[:, a] - tgt_joints[:, b], dim=1)
        total = total + torch.abs(out_len - tgt_len).mean()
    return total / max(len(joint_pairs), 1)


#representative bone pairs for the 33-joint MediaPipe skeleton
MEDIAPIPE_BONE_PAIRS = [
    (11, 12),  #shoulders
    (11, 13), (13, 15),  #left arm
    (12, 14), (14, 16),  #right arm
    (11, 23), (12, 24),  #torso sides
    (23, 24),            #hip bar
    (23, 25), (25, 27), (27, 29), (29, 31),  #left leg
    (24, 26), (26, 28), (28, 30), (30, 32),  #right leg
]


#------------------------------------------------------------
#TRAINING SETUP
#------------------------------------------------------------

#prepare clean vectors
clean_vectors = np.array([p["vector"] for p in poses_db.values() if p["ok"]])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

noisy_inputs, clean_targets = generate_noisy_dataset(clean_vectors, noise_factor=0.03, occlusion_prob=0.05)
dataset = TensorDataset(noisy_inputs, clean_targets)
loader = DataLoader(dataset, batch_size=16, shuffle=True, drop_last=True)  #drop_last avoids batch_size=1 edge cases with LayerNorm

#build the weight vector once and keep it on device
joint_weights = build_joint_weight_vector(peripheral_weight=3.0, device=device)

#re-initialize the improved architecture
model = RobustPoseAE(input_dim=99, latent_dim=64).to(device)

#AdamW with a lower base LR: the skip connection already accelerates convergence
optimizer = optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-4)

#cosine annealing avoids the optimizer overshooting the narrow loss basin of correct limb geometry
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150, eta_min=1e-5)


#------------------------------------------------------------
#TRAINING LOOP
#------------------------------------------------------------

model.train()
epochs = 150
print("Training RobustPoseAE with peripheral-weighted loss and bone consistency term...")

for epoch in range(epochs):
    total_loss = 0.0
    total_wmse = 0.0
    total_var  = 0.0
    total_bone = 0.0

    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()

        output = model(batch_x)

        #primary reconstruction loss with 3x peripheral joint emphasis
        loss_wmse = weighted_mse(output, batch_y, joint_weights)

        #per-joint variance penalty: forces limb spread to match instructor variance
        loss_var = per_joint_variance_loss(output, batch_y)

        #bone length consistency: prevents skeleton distortion in normalized space
        loss_bone = bone_length_consistency_loss(output, batch_y, MEDIAPIPE_BONE_PAIRS)

        #combined loss with tuned coefficients
        #wmse drives accuracy; var_loss drives spread; bone_loss prevents geometric distortion
        loss = loss_wmse + 0.3 * loss_var + 0.1 * loss_bone

        loss.backward()
        #gradient clipping prevents weight explosion in the skip-residual paths
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_wmse += loss_wmse.item()
        total_var  += loss_var.item()
        total_bone += loss_bone.item()

    scheduler.step()

    if (epoch + 1) % 25 == 0 or epoch == 0:
        n = len(loader)
        print(
            f"Epoch [{epoch+1:3d}/{epochs}] "
            f"Total={total_loss/n:.5f} | "
            f"W-MSE={total_wmse/n:.5f} | "
            f"VarLoss={total_var/n:.5f} | "
            f"BoneLoss={total_bone/n:.5f} | "
            f"LR={scheduler.get_last_lr()[0]:.6f}"
        )

print("Training complete. Peripheral joints are now weighted 3x in optimization.")

Training updated robust autoencoder to prevent point collapse...
Epoch [1/150] -> Loss: 0.055009
Epoch [25/150] -> Loss: 0.020197
Epoch [50/150] -> Loss: 0.017360
Epoch [75/150] -> Loss: 0.016199
Epoch [100/150] -> Loss: 0.015378
Epoch [125/150] -> Loss: 0.015249
Epoch [150/150] -> Loss: 0.014879
Robust model training finished. Ready to re-run metrics and dashboards.


In [74]:
results_ae = {}
model.eval()

print("Recalculating dataset metrics using the trained autoencoder filter...")

with torch.no_grad():
    for cat in categories:
        inst_pose = data[cat]["instructor_pose"]
        results_ae[cat] = []

        for student_path, student_pose in zip(data[cat]["students"], data[cat]["students_poses"]):
            name = os.path.basename(student_path)

            #pass the raw student coordinates through the network to filter noise
            student_tensor = torch.tensor(student_pose["vector"], dtype=torch.float32).unsqueeze(0).to(device)
            corrected_tensor = model(student_tensor)
            corrected_vector = corrected_tensor.cpu().squeeze(0).numpy()

            #calculate how much the network altered the pose
            intrinsic_error = np.mean((student_pose["vector"] - corrected_vector) ** 2)

            #compute cosine similarity between instructor and the corrected user pose
            cos_sim = cosine_similarity(inst_pose["vector"], corrected_vector)
            
            #keep original joint angle calculations for consistency in visualization
            rmse = rmse_angles(inst_pose["angles"], student_pose["angles"])
            mae  = mae_angles(inst_pose["angles"], student_pose["angles"])
            pct  = pct_angles_right(inst_pose["angles"], student_pose["angles"], umbral=15)
            errors = error_per_articulation(inst_pose["angles"], student_pose["angles"])

            #escaped quotes fixed below by removing trailing backslashes
            results_ae[cat].append({
                "student":            name,
                "Similitud coseno ": round(cos_sim, 4),
                "Error Intrínseco AE ↓": round(intrinsic_error, 5),
                "RMSE angles ":    round(rmse, 2) if not np.isnan(rmse) else np.nan,
                "MAE angles ":     round(mae, 2)  if not np.isnan(mae)  else np.nan,
                "% correct (±15°) ": pct,
                "errores_detalle":   errors,
            })

#overwrite global results object so subsequent plotting cells read the filtered entries
results = results_ae
print("Metrics updated. You can now re-run the downstream visualization cells.")

Recalculating dataset metrics using the trained autoencoder filter...
Metrics updated. You can now re-run the downstream visualization cells.


In [75]:
#extract a single sample to visualize the autoencoder effect
sample_cat = categories[0]
sample_student_pose = data[sample_cat]["students_poses"][0]

#prepare data for the network
raw_vector = sample_student_pose["vector"]
student_tensor = torch.tensor(raw_vector, dtype=torch.float32).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    corrected_tensor = model(student_tensor)
    corrected_vector = corrected_tensor.cpu().squeeze(0).numpy()

#reshape back to coordinate matrices (33 keypoints, 3 dimensions)
raw_coords = raw_vector.reshape(33, 3)
corrected_coords = corrected_vector.reshape(33, 3)

print("Original raw coordinates sample (first 3 keypoints):")
print(raw_coords[:3])
print("\nFiltered autoencoder coordinates sample (first 3 keypoints):")
print(corrected_coords[:3])

Original raw coordinates sample (first 3 keypoints):
[[-0.07864183 -0.01642911 -0.05594031]
 [-0.10580549 -0.01760357 -0.1170238 ]
 [-0.10661894 -0.0183542  -0.11775969]]

Filtered autoencoder coordinates sample (first 3 keypoints):
[[-0.08512276 -0.0895344  -0.03537503]
 [-0.10128147 -0.08811647 -0.09655574]
 [-0.10143325 -0.09268606 -0.09724458]]


In [87]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

#------------------------------------------------------------
#MEDIAPIPE 33-JOINT INDEX ANATOMY MAP
#peripheral limb joints (hands, feet, ankles, wrists) are the
#ones that collapse first under standard MSE — assign them 3x weight
#central torso joints (hips, shoulders, nose) stay at 1x weight
#------------------------------------------------------------

#indices for peripheral extremity joints in the 33-joint schema
PERIPHERAL_JOINT_INDICES = [
    15, 16,         #wrists
    17, 18, 19, 20, #pinky/index knuckles
    21, 22,         #thumbs
    27, 28,         #ankles
    29, 30,         #heels
    31, 32          #foot index
]

#all remaining joints treated as central/torso weight group
CENTRAL_JOINT_INDICES = [i for i in range(33) if i not in PERIPHERAL_JOINT_INDICES]

def build_joint_weight_vector(peripheral_weight=3.0, device="cpu"):
    #expand joint-level weights to cover all 3 coordinates (x, y, z) per joint
    weights = torch.ones(99, dtype=torch.float32)
    for ji in PERIPHERAL_JOINT_INDICES:
        base = ji * 3
        weights[base]     = peripheral_weight   #x
        weights[base + 1] = peripheral_weight   #y
        weights[base + 2] = peripheral_weight   #z
    return weights.to(device)


def weighted_mse(output, target, weights):
    #element-wise squared error scaled by the per-coordinate weight vector
    sq_err = (output - target) ** 2
    return (sq_err * weights).mean()


def per_joint_variance_loss(output, target):
    #reshape to (batch, 33, 3) to compute variance per joint independently
    #this prevents coarse global std from being satisfied by central joints alone
    out_joints = output.view(-1, 33, 3)
    tgt_joints = target.view(-1, 33, 3)
    #variance per joint across the batch dimension: shape (33, 3)
    out_var = out_joints.var(dim=0)
    tgt_var = tgt_joints.var(dim=0)
    #penalize any joint whose output variance is far below the target variance
    #clamp prevents penalizing overconfident joints that are already well-reconstructed
    return torch.clamp(tgt_var - out_var, min=0).mean()


def bone_length_consistency_loss(output, target, joint_pairs):
    #optional structural loss: penalizes segment length distortion between adjacent joints
    #joint_pairs is a list of (idx_a, idx_b) tuples for connected bones
    out_joints = output.view(-1, 33, 3)
    tgt_joints = target.view(-1, 33, 3)
    total = 0.0
    for a, b in joint_pairs:
        out_len = torch.norm(out_joints[:, a] - out_joints[:, b], dim=1)
        tgt_len = torch.norm(tgt_joints[:, a] - tgt_joints[:, b], dim=1)
        total = total + torch.abs(out_len - tgt_len).mean()
    return total / max(len(joint_pairs), 1)


#representative bone pairs for the 33-joint MediaPipe skeleton
MEDIAPIPE_BONE_PAIRS = [
    (11, 12),  #shoulders
    (11, 13), (13, 15),  #left arm
    (12, 14), (14, 16),  #right arm
    (11, 23), (12, 24),  #torso sides
    (23, 24),            #hip bar
    (23, 25), (25, 27), (27, 29), (29, 31),  #left leg
    (24, 26), (26, 28), (28, 30), (30, 32),  #right leg
]


#------------------------------------------------------------
#TRAINING SETUP
#------------------------------------------------------------

#prepare clean vectors
clean_vectors = np.array([p["vector"] for p in poses_db.values() if p["ok"]])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

noisy_inputs, clean_targets = generate_noisy_dataset(clean_vectors, noise_factor=0.03, occlusion_prob=0.05)
dataset = TensorDataset(noisy_inputs, clean_targets)
loader = DataLoader(dataset, batch_size=16, shuffle=True, drop_last=True)  #drop_last avoids batch_size=1 edge cases with LayerNorm

#build the weight vector once and keep it on device
joint_weights = build_joint_weight_vector(peripheral_weight=3.0, device=device)

#re-initialize the improved architecture
model = RobustPoseAE(input_dim=99, latent_dim=64).to(device)

#AdamW with a lower base LR: the skip connection already accelerates convergence
optimizer = optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-4)

#cosine annealing avoids the optimizer overshooting the narrow loss basin of correct limb geometry
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150, eta_min=1e-5)


#------------------------------------------------------------
#TRAINING LOOP
#------------------------------------------------------------

model.train()
epochs = 150
print("Training RobustPoseAE with peripheral-weighted loss and bone consistency term...")

for epoch in range(epochs):
    total_loss = 0.0
    total_wmse = 0.0
    total_var  = 0.0
    total_bone = 0.0

    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()

        output = model(batch_x)

        #primary reconstruction loss with 3x peripheral joint emphasis
        loss_wmse = weighted_mse(output, batch_y, joint_weights)

        #per-joint variance penalty: forces limb spread to match instructor variance
        loss_var = per_joint_variance_loss(output, batch_y)

        #bone length consistency: prevents skeleton distortion in normalized space
        loss_bone = bone_length_consistency_loss(output, batch_y, MEDIAPIPE_BONE_PAIRS)

        #combined loss with tuned coefficients
        #wmse drives accuracy; var_loss drives spread; bone_loss prevents geometric distortion
        loss = loss_wmse + 0.3 * loss_var + 0.1 * loss_bone

        loss.backward()
        #gradient clipping prevents weight explosion in the skip-residual paths
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_wmse += loss_wmse.item()
        total_var  += loss_var.item()
        total_bone += loss_bone.item()

    scheduler.step()

    if (epoch + 1) % 25 == 0 or epoch == 0:
        n = len(loader)
        print(
            f"Epoch [{epoch+1:3d}/{epochs}] "
            f"Total={total_loss/n:.5f} | "
            f"W-MSE={total_wmse/n:.5f} | "
            f"VarLoss={total_var/n:.5f} | "
            f"BoneLoss={total_bone/n:.5f} | "
            f"LR={scheduler.get_last_lr()[0]:.6f}"
        )

print("Training complete. Peripheral joints are now weighted 3x in optimization.")

Training RobustPoseAE with peripheral-weighted loss and bone consistency term...
Epoch [  1/150] Total=0.12463 | W-MSE=0.10183 | VarLoss=0.01974 | BoneLoss=0.16875 | LR=0.000800
Epoch [ 25/150] Total=0.03600 | W-MSE=0.02599 | VarLoss=0.00527 | BoneLoss=0.08428 | LR=0.000747
Epoch [ 50/150] Total=0.03136 | W-MSE=0.02229 | VarLoss=0.00441 | BoneLoss=0.07750 | LR=0.000603
Epoch [ 75/150] Total=0.02842 | W-MSE=0.01995 | VarLoss=0.00402 | BoneLoss=0.07263 | LR=0.000405
Epoch [100/150] Total=0.02731 | W-MSE=0.01904 | VarLoss=0.00370 | BoneLoss=0.07152 | LR=0.000207
Epoch [125/150] Total=0.02598 | W-MSE=0.01797 | VarLoss=0.00382 | BoneLoss=0.06864 | LR=0.000063
Epoch [150/150] Total=0.02594 | W-MSE=0.01800 | VarLoss=0.00353 | BoneLoss=0.06879 | LR=0.000010
Training complete. Peripheral joints are now weighted 3x in optimization.


In [88]:
import os
import torch
import numpy as np

results_ae = {}
model.eval()

#collapse detection threshold: if output std falls below this the network is still mean-collapsing
COLLAPSE_STD_THRESHOLD = 0.05

print("Recalculating dataset metrics using the trained autoencoder filter...")

with torch.no_grad():
    for cat in categories:
        inst_pose = data[cat]["instructor_pose"]
        results_ae[cat] = []

        for student_path, student_pose in zip(data[cat]["students"], data[cat]["students_poses"]):
            name = os.path.basename(student_path)

            student_tensor = torch.tensor(student_pose["vector"], dtype=torch.float32).unsqueeze(0).to(device)
            corrected_tensor = model(student_tensor)
            corrected_vector = corrected_tensor.cpu().squeeze(0).numpy()

            #guard: detect if the model is still producing collapsed output for this sample
            output_std = corrected_vector.std()
            if output_std < COLLAPSE_STD_THRESHOLD:
                #fallback to raw vector instead of silently propagating a collapsed skeleton
                print(f"[WARN] collapse detected for {name} (std={output_std:.4f}), falling back to raw vector")
                corrected_vector = student_pose["vector"].copy()

            intrinsic_error = np.mean((student_pose["vector"] - corrected_vector) ** 2)
            cos_sim = cosine_similarity(inst_pose["vector"], corrected_vector)
            rmse = rmse_angles(inst_pose["angles"], student_pose["angles"])
            mae  = mae_angles(inst_pose["angles"], student_pose["angles"])
            pct  = pct_angles_right(inst_pose["angles"], student_pose["angles"], umbral=15)
            errors = error_per_articulation(inst_pose["angles"], student_pose["angles"])

            results_ae[cat].append({
                "student":                name,
                "Similitud coseno":       round(cos_sim, 4),
                "Error Intrínseco AE":    round(intrinsic_error, 5),
                "RMSE angles":            round(rmse, 2) if not np.isnan(rmse) else np.nan,
                "MAE angles":             round(mae, 2)  if not np.isnan(mae)  else np.nan,
                "% correct (±15°)":       pct,
                "errores_detalle":        errors,
                "ae_std":                 round(float(output_std), 4),  #track collapse metric per student
            })

results = results_ae
print("Metrics updated. You can now re-run the downstream visualization cells.")

Recalculating dataset metrics using the trained autoencoder filter...
Metrics updated. You can now re-run the downstream visualization cells.


In [94]:
import random
import cv2
import os
import base64
from PIL import Image
import io
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def visualize_complete_ae_dashboard_fixed_final_v2(random_sample=True):
    #1. pick a category and student path
    if random_sample:
        selected_cat = random.choice(categories)
        student_path = random.choice(data[selected_cat]["students"])
    else:
        selected_cat = categories[0]
        student_path = data[selected_cat]["students"][0]
        
    #2. read the original image dimensions safely using PIL
    try:
        pil_img = Image.open(student_path)
        w_img, h_img = pil_img.size
        
        #convert image to base64 string format for absolute web rendering
        buffered = io.BytesIO()
        pil_img.save(buffered, format="PNG")
        img_base64 = base64.b64encode(buffered.getvalue()).decode()
        formatted_url = f"data:image/png;base64,{img_base64}"
    except Exception as e:
        print(f"Error loading or converting image: {e}")
        return

    #3. find the corresponding pose data in poses_db
    student_pose = poses_db[student_path]
    raw_vector = student_pose["vector"]
    
    #4. run the autoencoder inference
    student_tensor = torch.tensor(raw_vector, dtype=torch.float32).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        corrected_tensor = model(student_tensor)
        corrected_vector = corrected_tensor.cpu().squeeze(0).numpy()

    #5. build explicit (33, 2) coordinate tracking sets using strict stride unpacking
    #this completely eliminates the Z axis leakage flatten error
    raw_x = raw_vector[0::3]
    raw_y = raw_vector[1::3]
    raw_coords_2d = np.stack([raw_x, raw_y], axis=1)
    
    corr_x = corrected_vector[0::3]
    corr_y = corrected_vector[1::3]
    corr_coords_2d = np.stack([corr_x, corr_y], axis=1)
    
    #6. calculate the original image coordinate mappings accurately
    img_cv = cv2.imread(student_path)
    img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    detection_result = POSE_MODEL.detect(mp_image)
    
    if not detection_result.pose_landmarks:
        print("Error: could not re-run detection for pixel alignment mapping.")
        return
        
    #extract original raw points before centering and scaling
    orig_landmarks = detection_result.pose_landmarks[0]
    raw_pixel_coords = np.array([[l.x * w_img, l.y * h_img] for l in orig_landmarks])
    
    #compute absolute min/max bounding box of the true visible skeleton
    min_x, max_x = raw_pixel_coords[:, 0].min(), raw_pixel_coords[:, 0].max()
    min_y, max_y = raw_pixel_coords[:, 1].min(), raw_pixel_coords[:, 1].max()
    
    #map the normalized autoencoder outputs directly to the true skeletal bounding box
    corr_pixel_coords = np.zeros((33, 2))
    
    #normalize the strides to avoid projection collapse bounds
    norm_x_scaled = (corr_x - corr_x.min()) / (corr_x.max() - corr_x.min() + 1e-8)
    norm_y_scaled = (corr_y - corr_y.min()) / (corr_y.max() - corr_y.min() + 1e-8)
    
    corr_pixel_coords[:, 0] = min_x + norm_x_scaled * (max_x - min_x)
    corr_pixel_coords[:, 1] = min_y + norm_y_scaled * (max_y - min_y)

    #7. custom baseline helper function to draw bones on a blank clean layout
    def draw_skeleton_clean(coords_2d, joint_color):
        traces = []
        for a_key, b_key in JOINT_PAIRS:
            ai, bi = LM[a_key], LM[b_key]
            a, b = coords_2d[ai], coords_2d[bi]
            traces.append(go.Scatter(
                x=[a[0], b[0]], y=[-a[1], -b[1]], #y-flip matching standard graph spaces
                mode="lines", line=dict(color=joint_color, width=2.5),
                showlegend=False, hoverinfo="skip"
            ))
        traces.append(go.Scatter(
            x=coords_2d[:, 0], y=-coords_2d[:, 1],
            mode="markers", marker=dict(size=5, color="#ff7f0e"),
            showlegend=False, hoverinfo="skip"
        ))
        return traces

    #8. custom overlay helper function to project bones directly on image canvas pixels
    def draw_skeleton_overlay(pixel_coords, joint_color):
        traces = []
        for a_key, b_key in JOINT_PAIRS:
            ai, bi = LM[a_key], LM[b_key]
            a, b = pixel_coords[ai], pixel_coords[bi]
            traces.append(go.Scatter(
                x=[a[0], b[0]], y=[h_img - a[1], h_img - b[1]],
                mode="lines", line=dict(color=joint_color, width=3.5),
                showlegend=False, hoverinfo="skip"
            ))
        traces.append(go.Scatter(
            x=pixel_coords[:, 0], y=h_img - pixel_coords[:, 1],
            mode="markers", marker=dict(size=5, color="yellow"),
            showlegend=False, hoverinfo="skip"
        ))
        return traces

    #9. configure a 2x3 grid matrix to partition all 5 separate visualization slots
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=(
            "A. Clean Original Image", "B. Standalone Raw Skeleton", "C. Standalone AE Skeleton",
            "D. Raw Skeleton Overlay", "E. AE Skeleton Overlay", ""
        ),
        vertical_spacing=0.15,
        horizontal_spacing=0.08
    )

    #10. setup background image target parameters using the base64 source string
    img_slots = [(1, 1), (2, 1), (2, 2)]
    for r, c in img_slots:
        fig.add_layout_image(
            dict(
                source=formatted_url,
                xref="x", yref="y",
                x=0, y=h_img,
                sizex=w_img, sizey=h_img,
                sizing="stretch",
                opacity=1.0,
                layer="below"
            ),
            row=r, col=c
        )

    #11. render standalone geometry tracers into row 1 using isolated 2D matrices
    for t in draw_skeleton_clean(raw_coords_2d, "indianred"):
        fig.add_trace(t, row=1, col=2)
        
    for t in draw_skeleton_clean(corr_coords_2d, "mediumseagreen"):
        fig.add_trace(t, row=1, col=3)

    #12. render spatial overlay tracers into row 2
    for t in draw_skeleton_overlay(raw_pixel_coords, "red"):
        fig.add_trace(t, row=2, col=1)
        
    for t in draw_skeleton_overlay(corr_pixel_coords, "cyan"):
        fig.add_trace(t, row=2, col=2)

    #13. apply bounding boxes to standalone graphs based directly on their real 2D ranges
    #this completely stops the isolated points from collapsing into dots
    for c, coords in [(2, raw_coords_2d), (3, corr_coords_2d)]:
        x_min, x_max = coords[:, 0].min(), coords[:, 0].max()
        y_min, y_max = coords[:, 1].min(), coords[:, 1].max()
        #add a small padding margin so bones do not touch the edges
        pad_x = (x_max - x_min) * 0.15 + 1e-5
        pad_y = (y_max - y_min) * 0.15 + 1e-5
        
        fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, range=[x_min - pad_x, x_max + pad_x], row=1, col=c)
        fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, range=[-y_max - pad_y, -y_min + pad_y], scaleanchor=f"x{c}", scaleratio=1, row=1, col=c)
        
    #canvas tracking targets use absolute pixel dimension boundaries
    for r, c in img_slots:
        axis_idx = "" if (r == 1 and c == 1) else f"{3 if (r == 2 and c == 1) else 4}"
        fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, range=[0, w_img], row=r, col=c)
        fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, range=[0, h_img], scaleanchor=f"x{axis_idx}", scaleratio=1, row=r, col=c)

    #hide axis and borders completely on the last unused cell (row 2, col 3)
    fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, visible=False, row=2, col=3)
    fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, visible=False, row=2, col=3)

    #14. finalize layout dimensions and render
    fig.update_layout(
        title=f"Complete Multi-View Pose Verification — File: {os.path.basename(student_path)}",
        height=850,
        showlegend=False
    )
    fig.show()

#run the final strides-corrected visualization dashboard
visualize_complete_ae_dashboard_fixed_final_v2(random_sample=True)

In [95]:
#--------

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os
import torch

#select the first category to visualize
selected_cat = categories[0]
inst_pose = data[selected_cat]["instructor_pose"]

students      = data[selected_cat]["students"]
student_poses = data[selected_cat]["students_poses"]

names            = [os.path.basename(p)[:25] for p in students]
raw_cos_scores   = []
ae_cos_scores    = []
intrinsic_errors = []

#collapse detection threshold reused from cell 3
COLLAPSE_STD_THRESHOLD = 0.05

#set to True to auto-correct left/right orientation mismatches
handle_mirroring = True

model.eval()
with torch.no_grad():
    for student_pose in student_poses:
        inst_vec = inst_pose["vector"]
        raw_vec  = student_pose["vector"]

        #pass raw vector through the trained AE
        student_tensor   = torch.tensor(raw_vec, dtype=torch.float32).unsqueeze(0).to(device)
        corrected_tensor = model(student_tensor)
        corrected_vector = corrected_tensor.cpu().squeeze(0).numpy()

        #collapse guard: fall back to raw if the AE output collapsed
        if corrected_vector.std() < COLLAPSE_STD_THRESHOLD:
            corrected_vector = raw_vec.copy()

        if handle_mirroring:
            raw_coords  = raw_vec.reshape(33, 3)
            corr_coords = corrected_vector.reshape(33, 3)

            #build x-flipped versions for both raw and corrected
            raw_flipped      = raw_coords.copy()
            raw_flipped[:,0] = -raw_flipped[:,0]
            raw_flipped_vec  = raw_flipped.flatten()

            corr_flipped      = corr_coords.copy()
            corr_flipped[:,0] = -corr_flipped[:,0]
            corr_flipped_vec  = corr_flipped.flatten()

            #choose the orientation that best matches the instructor
            if cosine_similarity(inst_vec, raw_flipped_vec) > cosine_similarity(inst_vec, raw_vec):
                raw_sim = cosine_similarity(inst_vec, raw_flipped_vec)
                ae_sim  = cosine_similarity(inst_vec, corr_flipped_vec)
            else:
                raw_sim = cosine_similarity(inst_vec, raw_vec)
                ae_sim  = cosine_similarity(inst_vec, corrected_vector)
        else:
            raw_sim = cosine_similarity(inst_vec, raw_vec)
            ae_sim  = cosine_similarity(inst_vec, corrected_vector)

        raw_cos_scores.append(raw_sim)
        ae_cos_scores.append(ae_sim)

        #intrinsic reconstruction error (MSE between raw input and AE output)
        rec_error = np.mean((raw_vec - corrected_vector) ** 2)
        intrinsic_errors.append(rec_error)

#------------------------------------------------------------
#FIGURE: 1 row x 2 cols
#left:  grouped bars Raw vs AE cosine similarity per student
#right: intrinsic reconstruction error per student
#------------------------------------------------------------
fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=0.15,
    subplot_titles=(
        "Cosine Similarity — Raw vs AE Filtered",
        "AE Intrinsic Reconstruction Error (MSE)"
    )
)

#grouped bars: raw scores
fig.add_trace(
    go.Bar(
        x=names, y=raw_cos_scores,
        name="Raw Match Score",
        marker_color="indianred",
        text=[f"{v:.3f}" for v in raw_cos_scores],
        textposition="outside"
    ),
    row=1, col=1
)

#grouped bars: AE filtered scores
fig.add_trace(
    go.Bar(
        x=names, y=ae_cos_scores,
        name="AE Filtered Match Score",
        marker_color="mediumseagreen",
        text=[f"{v:.3f}" for v in ae_cos_scores],
        textposition="outside"
    ),
    row=1, col=1
)

#intrinsic error bars
fig.add_trace(
    go.Bar(
        x=names, y=intrinsic_errors,
        name="Intrinsic Pose Deviation",
        marker_color="royalblue",
        text=[f"{v:.4f}" for v in intrinsic_errors],
        textposition="outside"
    ),
    row=1, col=2
)

#reference line at cosine similarity = 1.0 (perfect match)
fig.add_hline(
    y=1.0, line_dash="dot", line_color="gray",
    annotation_text="Perfect Match", annotation_position="top right",
    row=1, col=1
)

fig.update_layout(
    title=f"Autoencoder Impact Analysis — Category: {selected_cat}",
    barmode="group",
    height=550,
    showlegend=True,
    margin=dict(b=160, t=80),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_xaxes(tickangle=45, row=1, col=2)

#y-axis ranges
fig.update_yaxes(range=[0, 1.15], title_text="Cosine Similarity", row=1, col=1)
fig.update_yaxes(title_text="MSE", row=1, col=2)

fig.show()

In [109]:
import random
import cv2
import os
import base64
import io
import torch
import numpy as np
import mediapipe as mp
from PIL import Image
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#collapse detection threshold: AE outputs below this std are considered collapsed
COLLAPSE_STD_THRESHOLD = 0.05

def visualize_complete_ae_dashboard(random_sample=True):

    #------------------------------------------------------------------
    #1. pick a category and student path
    #------------------------------------------------------------------
    if random_sample:
        selected_cat = random.choice(categories)
        student_path = random.choice(data[selected_cat]["students"])
    else:
        selected_cat = categories[0]
        student_path = data[selected_cat]["students"][0]

    #------------------------------------------------------------------
    #2. load image and convert to base64 for plotly layout_image
    #------------------------------------------------------------------
    try:
        pil_img   = Image.open(student_path)
        w_img, h_img = pil_img.size
        buffered  = io.BytesIO()
        pil_img.save(buffered, format="PNG")
        img_b64   = base64.b64encode(buffered.getvalue()).decode()
        img_url   = f"data:image/png;base64,{img_b64}"
    except Exception as e:
        print(f"Error loading image: {e}")
        return

    #------------------------------------------------------------------
    #3. get the stored normalized vector for this student
    #------------------------------------------------------------------
    student_pose = poses_db[student_path]
    raw_vector   = student_pose["vector"]           #shape: (99,) normalized

    #------------------------------------------------------------------
    #4. run AE inference
    #------------------------------------------------------------------
    student_tensor   = torch.tensor(raw_vector, dtype=torch.float32).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        corrected_tensor = model(student_tensor)
        corrected_vector = corrected_tensor.cpu().squeeze(0).numpy()

    #collapse guard: replace with raw if the AE output collapsed
    ae_collapsed = corrected_vector.std() < COLLAPSE_STD_THRESHOLD
    if ae_collapsed:
        print(f"[WARN] AE collapse detected for {os.path.basename(student_path)}, using raw fallback for display.")
        corrected_vector = raw_vector.copy()

    #------------------------------------------------------------------
    #5. extract 2D coordinates for STANDALONE skeleton panels (B and C)
    #
    #ROOT CAUSE FIX FOR PANEL C (single-dot collapse):
    #Both raw_vector and corrected_vector are in the same normalized space
    #(centered around 0, scaled to [-1, 1] approximately).
    #We draw them using their own per-axis min/max range so both panels
    #use identical axis logic. The previous code drew corr_coords_2d without
    #rescaling, causing all points to cluster at near-zero center values.
    #No coordinate remapping is needed — just let the axis range auto-fit.
    #------------------------------------------------------------------
    raw_x         = raw_vector[0::3]
    raw_y         = raw_vector[1::3]
    raw_coords_2d = np.stack([raw_x, raw_y], axis=1)       #(33,2) normalized

    corr_x         = corrected_vector[0::3]
    corr_y         = corrected_vector[1::3]
    corr_coords_2d = np.stack([corr_x, corr_y], axis=1)    #(33,2) normalized

    #------------------------------------------------------------------
    #6. re-run MediaPipe to get pixel-space raw landmarks for overlay panels
    #------------------------------------------------------------------
    img_cv   = cv2.imread(student_path)
    img_rgb  = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    detection_result = POSE_MODEL.detect(mp_image)

    if not detection_result.pose_landmarks:
        print("Error: MediaPipe could not detect landmarks for pixel alignment.")
        return

    orig_landmarks   = detection_result.pose_landmarks[0]
    #raw pixel coordinates directly from MediaPipe (no normalization)
    raw_pixel_coords = np.array([[l.x * w_img, l.y * h_img] for l in orig_landmarks])  #(33,2)

    #------------------------------------------------------------------
    #7. project AE corrected vector onto pixel space for overlay panel E
    #
    #ROOT CAUSE FIX FOR PANEL E (scale mismatch):
    #The previous code independently min-max scaled corr_x and corr_y
    #using the corrected vector's own range, then mapped to the bounding box
    #of raw_pixel_coords. This breaks scale consistency because the AE output
    #may have a different range than raw_vector in each axis independently.
    #
    #CORRECT APPROACH:
    #Both raw_vector and corrected_vector live in the same normalized space
    #because the AE was trained on normalized vectors. Therefore we must:
    #  a) Compute the bounding box from raw_pixel_coords (the ground truth pixel space).
    #  b) Use the RAW vector's own min/max as the normalization reference frame
    #     for BOTH raw and corrected projections.
    #This guarantees that both skeletons are mapped through the same linear
    #transform, so relative joint positions are preserved and comparable.
    #------------------------------------------------------------------
    min_px, max_px = raw_pixel_coords[:, 0].min(), raw_pixel_coords[:, 0].max()
    min_py, max_py = raw_pixel_coords[:, 1].min(), raw_pixel_coords[:, 1].max()

    #use raw_x and raw_y ranges as the shared normalization reference
    raw_x_min, raw_x_max = raw_x.min(), raw_x.max()
    raw_y_min, raw_y_max = raw_y.min(), raw_y.max()

    def norm_to_pixel_x(nx):
        #maps a value in raw normalized x-space to pixel x-space linearly
        t = (nx - raw_x_min) / (raw_x_max - raw_x_min + 1e-8)
        return min_px + t * (max_px - min_px)

    def norm_to_pixel_y(ny):
        #maps a value in raw normalized y-space to pixel y-space linearly
        t = (ny - raw_y_min) / (raw_y_max - raw_y_min + 1e-8)
        return min_py + t * (max_py - min_py)

    corr_pixel_coords = np.stack([
        norm_to_pixel_x(corr_x),
        norm_to_pixel_y(corr_y)
    ], axis=1)  #(33,2) in pixel space using shared reference frame

    #------------------------------------------------------------------
    #8. skeleton drawing helpers
    #------------------------------------------------------------------
    def draw_skeleton_clean(coords_2d, bone_color):
        #draws bones and joints on a blank normalized coordinate canvas
        traces = []
        for a_key, b_key in JOINT_PAIRS:
            ai, bi = LM[a_key], LM[b_key]
            a, b   = coords_2d[ai], coords_2d[bi]
            traces.append(go.Scatter(
                x=[a[0], b[0]], y=[-a[1], -b[1]],  #y-flip: image y goes down, graph y goes up
                mode="lines",
                line=dict(color=bone_color, width=2.5),
                showlegend=False, hoverinfo="skip"
            ))
        traces.append(go.Scatter(
            x=coords_2d[:, 0], y=-coords_2d[:, 1],
            mode="markers",
            marker=dict(size=5, color="#ff7f0e"),
            showlegend=False, hoverinfo="skip"
        ))
        return traces

    def draw_skeleton_overlay(pixel_coords, bone_color, dot_color="yellow"):
        #draws bones and joints projected onto the image pixel coordinate canvas
        traces = []
        for a_key, b_key in JOINT_PAIRS:
            ai, bi = LM[a_key], LM[b_key]
            a, b   = pixel_coords[ai], pixel_coords[bi]
            traces.append(go.Scatter(
                x=[a[0], b[0]], y=[h_img - a[1], h_img - b[1]],  #flip y for plotly image origin
                mode="lines",
                line=dict(color=bone_color, width=3.5),
                showlegend=False, hoverinfo="skip"
            ))
        traces.append(go.Scatter(
            x=pixel_coords[:, 0], y=h_img - pixel_coords[:, 1],
            mode="markers",
            marker=dict(size=5, color=dot_color),
            showlegend=False, hoverinfo="skip"
        ))
        return traces

    #------------------------------------------------------------------
    #9. build 2x3 subplot grid
    #------------------------------------------------------------------
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=(
            "A. Clean Original Image",
            "B. Standalone Raw Skeleton",
            "C. Standalone AE Skeleton",
            "D. Raw Skeleton Overlay",
            "E. AE Skeleton Overlay",
            ""
        ),
        vertical_spacing=0.12,
        horizontal_spacing=0.06
    )

    #------------------------------------------------------------------
    #10. place background images on panels A, D, E
    #
    #Each layout_image must be bound to the correct xref/yref for its subplot cell.
    #Plotly assigns axis indices: (row=1,col=1)->x/y, (row=2,col=1)->x4/y4, (row=2,col=2)->x5/y5
    #We set these explicitly to avoid the wrong image appearing in the wrong panel.
    #------------------------------------------------------------------
    image_axis_map = {
        (1, 1): ("x",  "y" ),    #panel A
        (2, 1): ("x4", "y4"),    #panel D
        (2, 2): ("x5", "y5"),    #panel E
    }
    for (r, c), (xref, yref) in image_axis_map.items():
        fig.add_layout_image(dict(
            source  = img_url,
            xref    = xref, yref = yref,
            x=0, y=h_img,
            sizex   = w_img, sizey = h_img,
            sizing  = "stretch",
            opacity = 1.0,
            layer   = "below"
        ))

    #------------------------------------------------------------------
    #11. add standalone skeleton traces to panels B and C
    #------------------------------------------------------------------
    for trace in draw_skeleton_clean(raw_coords_2d, "indianred"):
        fig.add_trace(trace, row=1, col=2)

    for trace in draw_skeleton_clean(corr_coords_2d, "mediumseagreen"):
        fig.add_trace(trace, row=1, col=3)

    #------------------------------------------------------------------
    #12. add overlay skeleton traces to panels D and E
    #------------------------------------------------------------------
    for trace in draw_skeleton_overlay(raw_pixel_coords, "red", dot_color="yellow"):
        fig.add_trace(trace, row=2, col=1)

    for trace in draw_skeleton_overlay(corr_pixel_coords, "cyan", dot_color="yellow"):
        fig.add_trace(trace, row=2, col=2)

    #------------------------------------------------------------------
    #13. configure axes for standalone panels B and C
    #
    #FIX: compute the bounding box per panel from the actual coordinate data
    #and add padding so bones are never clipped to the panel edge.
    #Use scaleanchor to enforce equal aspect ratio (prevents skeleton distortion).
    #------------------------------------------------------------------
    for col_idx, coords in [(2, raw_coords_2d), (3, corr_coords_2d)]:
        x_min, x_max = coords[:, 0].min(), coords[:, 0].max()
        y_min, y_max = coords[:, 1].min(), coords[:, 1].max()
        pad_x = (x_max - x_min) * 0.18 + 1e-5
        pad_y = (y_max - y_min) * 0.18 + 1e-5

        fig.update_xaxes(
            showgrid=False, zeroline=False, showticklabels=False,
            range=[x_min - pad_x, x_max + pad_x],
            row=1, col=col_idx
        )
        #y is flipped in draw_skeleton_clean so range uses negated y values
        fig.update_yaxes(
            showgrid=False, zeroline=False, showticklabels=False,
            range=[-y_max - pad_y, -y_min + pad_y],
            scaleanchor=f"x{col_idx}",     #equal aspect ratio per panel
            scaleratio=1,
            row=1, col=col_idx
        )

    #------------------------------------------------------------------
    #14. configure axes for image-backed panels A, D, E
    #
    #Axis ranges must exactly match the pixel dimensions of the image
    #so that layout_image fills the entire panel without stretching or offset.
    #------------------------------------------------------------------
    for r, c in [(1, 1), (2, 1), (2, 2)]:
        fig.update_xaxes(
            showgrid=False, zeroline=False, showticklabels=False,
            range=[0, w_img],
            row=r, col=c
        )
        fig.update_yaxes(
            showgrid=False, zeroline=False, showticklabels=False,
            range=[0, h_img],
            row=r, col=c
        )

    #hide the empty sixth cell (row 2, col 3)
    fig.update_xaxes(visible=False, row=2, col=3)
    fig.update_yaxes(visible=False, row=2, col=3)

    #------------------------------------------------------------------
    #15. final layout and render
    #------------------------------------------------------------------
    ae_label = "AE COLLAPSED — raw fallback shown" if ae_collapsed else "AE Output"
    fig.update_layout(
        title=(
            f"Complete Multi-View Pose Verification — File: {os.path.basename(student_path)}<br>"
            f"<sup>Category: {selected_cat} | Panel C & E: {ae_label}</sup>"
        ),
        height=860,
        showlegend=False,
        margin=dict(t=100)
    )
    fig.show()


#run dashboard
visualize_complete_ae_dashboard(random_sample=True)